# 17. Four checks asked by the reviews

1. **Recalibration**: the outlet scores re-mapped by a line fitted inside the training folds (inner folds, the setting
   of step 7); the MAE before and after, and the difference from TF-IDF.
2. **Another population**: the MAE per level re-weighted to the NewsGuard levels of the political outlets that could
   not be collected.
3. **Leftover names**: the comparison without the outlets whose NewsGuard brand name is still in their text.
4. **Name only** (GPU): the frozen vector of the domain name alone, then a ridge over the outlets with the same folds.

Details in `robustness.py`; the table of the paper is `paper_robustness.csv`.

In [1]:
N_JOBS = 16
GPU_GB = 19
B = 10_000
NAME_PROBE = True                 # False: only the three checks on the CPU

In [2]:
# Parameters
N_JOBS = 8


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import metrics as ME
import paper as PA
import robustness as RB

art, outlets, rows = PA.load()
y, n = outlets.score.to_numpy(), len(outlets)
idx = PA.resamples(n, B)
draws = PA.draw_index(outlets, rows)
raw = {m: PA.matrix(PA.article_vector(m, art), draws, "all", n)[0] for m in RB.MODELS + ["LLM-zero-shot"]}
med = PA.baselines(outlets)["no text (median)"]
out = []                                   # one row per check, model, version and measure


def add(check, model, version, measure, x, ref=None):
    r = {"check": check, "model": model, "version": version, "measure": measure, "value": x[0],
         "low": np.percentile(x[1:], 2.5), "high": np.percentile(x[1:], 97.5)}
    if ref is not None:
        r.update(PA.compare(x, ref, measure))
    out.append(r)

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


## 1. Recalibration inside the training folds

In [4]:
cal = {m: RB.recalibrated(m, art, outlets, LOG).to_numpy() for m in raw}
V = {(m, v): ME.resampled_measures(p, y, idx, ["MAE", "Spearman"]) for v, P in [("raw", raw), ("recalibrated", cal)]
     for m, p in P.items()}
add("recalibration", "no text (median)", "raw", "MAE", ME.resampled_measures(med, y, idx, ["MAE"])["MAE"])
for (m, v), res in V.items():
    for mes, x in res.items():
        add("recalibration", m, v, mes, x, V[("TF-IDF", v)][mes] if m != "TF-IDF" else None)
pd.DataFrame(out).query("check == 'recalibration'").round(3)

,check,model,version,measure,value,low,high,diff,dlo,dhi,p
0,recalibration,no text (median),raw,MAE,22.126,19.947,24.372,NaN,NaN,NaN,NaN
1,recalibration,TF-IDF,raw,Spearman,0.674,0.604,0.733,NaN,NaN,NaN,NaN
2,recalibration,TF-IDF,raw,MAE,17.825,16.187,19.553,NaN,NaN,NaN,NaN
3,recalibration,Qwen3-8B,raw,Spearman,0.711,0.645,0.765,0.037,-0.003,0.079,0.069
4,recalibration,Qwen3-8B,raw,MAE,16.311,14.776,17.927,-1.514,-2.290,-0.743,0.000
5,recalibration,Mistral-24B,raw,Spearman,0.735,0.673,0.785,0.060,0.020,0.104,0.004
6,recalibration,Mistral-24B,raw,MAE,15.793,14.318,17.352,-2.031,-2.868,-1.159,0.000
7,recalibration,LLM-zero-shot,raw,Spearman,0.348,0.222,0.464,-0.326,-0.456,-0.195,0.000
8,recalibration,LLM-zero-shot,raw,MAE,24.377,22.531,26.221,6.552,4.959,8.129,0.000
9,recalibration,TF-IDF,recalibrated,Spearman,0.674,0.601,0.735,NaN,NaN,NaN,NaN


## 2. The MAE on another population of outlets

In [5]:
P2 = dict(raw, **{"no text (median)": med})
for pop, w in RB.level_weights(pd.read_csv(C.OUT / "paper_selection.csv")).items():
    R = {m: RB.reweighted_mae(p, outlets, w, idx[:2001]) for m, p in P2.items()}
    for m, x in R.items():
        add(f"MAE re-weighted: {pop}", m, "raw", "MAE", x, R["TF-IDF"] if m != "TF-IDF" else None)
pd.DataFrame(out)[lambda d: d.check.str.startswith("MAE re-weighted")].round(2)

,check,model,version,measure,value,low,high,diff,dlo,dhi,p
17,MAE re-weighted: not collected,TF-IDF,raw,MAE,22.89,20.85,24.98,NaN,NaN,NaN,NaN
18,MAE re-weighted: not collected,Qwen3-8B,raw,MAE,20.90,18.74,23.02,-1.99,-3.11,-0.96,0.00
19,MAE re-weighted: not collected,Mistral-24B,raw,MAE,20.17,18.02,22.33,-2.72,-3.69,-1.76,0.00
20,MAE re-weighted: not collected,LLM-zero-shot,raw,MAE,22.84,20.37,25.19,-0.05,-1.86,1.76,0.94
21,MAE re-weighted: not collected,no text (median),raw,MAE,34.93,33.03,36.71,12.03,10.73,13.34,0.00
22,MAE re-weighted: all political outlets,TF-IDF,raw,MAE,22.41,20.54,24.31,NaN,NaN,NaN,NaN
23,MAE re-weighted: all political outlets,Qwen3-8B,raw,MAE,20.44,18.45,22.37,-1.97,-3.00,-1.03,0.00
24,MAE re-weighted: all political outlets,Mistral-24B,raw,MAE,19.68,17.73,21.64,-2.73,-3.58,-1.87,0.00
25,MAE re-weighted: all political outlets,LLM-zero-shot,raw,MAE,24.03,21.75,26.25,1.62,-0.01,3.30,0.05
26,MAE re-weighted: all political outlets,no text (median),raw,MAE,32.55,30.85,34.15,10.15,8.97,11.34,0.00


## 3. Without the outlets whose brand name is left in the text

In [6]:
leaky = RB.outlets_with_brand(art)
ii = np.where(~outlets.index.isin(leaky))[0]
idx3 = ii[PA.resamples(len(ii), B)]
V3 = {m: ME.resampled_measures(p, y, idx3, ["MAE", "Spearman"]) for m, p in raw.items()}
for m, res in V3.items():
    for mes, x in res.items():
        add(f"without the {len(leaky)} outlets with their brand name", m, "raw", mes, x,
            V3["TF-IDF"][mes] if m != "TF-IDF" else None)
print(len(leaky), "outlets left out")
pd.DataFrame(out).to_csv(C.OUT / "paper_robustness.csv", index=False)      # saved now: the GPU may be busy
pd.DataFrame(out)[lambda d: d.check.str.startswith("without")].round(3)

16 outlets left out


,check,model,version,measure,value,low,high,diff,dlo,dhi,p
27,without the 16 outlets with their brand name,TF-IDF,raw,Spearman,0.689,0.619,0.746,NaN,NaN,NaN,NaN
28,without the 16 outlets with their brand name,TF-IDF,raw,MAE,17.967,16.251,19.726,NaN,NaN,NaN,NaN
29,without the 16 outlets with their brand name,Qwen3-8B,raw,Spearman,0.717,0.651,0.773,0.029,-0.009,0.069,0.142
30,without the 16 outlets with their brand name,Qwen3-8B,raw,MAE,16.536,14.918,18.201,-1.431,-2.198,-0.649,0.001
31,without the 16 outlets with their brand name,Mistral-24B,raw,Spearman,0.733,0.667,0.785,0.044,0.005,0.083,0.026
32,without the 16 outlets with their brand name,Mistral-24B,raw,MAE,16.155,14.612,17.753,-1.812,-2.655,-0.933,0.000
33,without the 16 outlets with their brand name,LLM-zero-shot,raw,Spearman,0.337,0.207,0.457,-0.351,-0.493,-0.218,0.000
34,without the 16 outlets with their brand name,LLM-zero-shot,raw,MAE,24.435,22.568,26.328,6.468,4.832,8.114,0.000


## 4. Name only: the frozen vector of the domain name, no article (GPU)

In [7]:
if NAME_PROBE:
    for m, X in RB.name_vectors(outlets, LOG, GPU_GB).items():
        for mes, x in ME.resampled_measures(RB.name_only(X, outlets), y, idx, ["MAE", "Spearman"]).items():
            add("name only", m, "raw", mes, x)
    display(pd.DataFrame(out).query("check == 'name only'").round(3))

2026-11-17 18:22:58  GPU: 10.0 GB free, 19 needed: waiting 5 minutes


2026-11-17 18:27:58  GPU: 8.8 GB free, 19 needed: waiting 5 minutes


2026-11-17 18:32:58  GPU: 0.3 GB free, 19 needed: waiting 5 minutes


2026-11-17 18:37:58  GPU: 1.6 GB free, 19 needed: waiting 5 minutes


2026-11-17 18:42:58  GPU: 0.3 GB free, 19 needed: waiting 5 minutes


2026-11-17 18:47:58  GPU: 0.3 GB free, 19 needed: waiting 5 minutes


... (92 outputs left out)

Loading checkpoint shards: 100%|██████████| 10/10 [01:12<00:00,  7.37s/it]

Loading checkpoint shards: 100%|██████████| 10/10 [01:12<00:00,  7.26s/it]

2026-11-18 01:09:50  Mistral: 2/226 articles


,check,model,version,measure,value,low,high,diff,dlo,dhi,p
35,name only,Qwen3-8B,raw,Spearman,0.345,0.228,0.453,NaN,NaN,NaN,NaN
36,name only,Qwen3-8B,raw,MAE,21.707,19.818,23.664,NaN,NaN,NaN,NaN
37,name only,Mistral-24B,raw,Spearman,0.385,0.269,0.491,NaN,NaN,NaN,NaN
38,name only,Mistral-24B,raw,MAE,20.408,18.470,22.421,NaN,NaN,NaN,NaN


In [8]:
R = pd.DataFrame(out)
R.to_csv(C.OUT / "paper_robustness.csv", index=False)
R.round(3)

,check,model,version,measure,value,low,high,diff,dlo,dhi,p
0,recalibration,no text (median),raw,MAE,22.126,19.947,24.372,NaN,NaN,NaN,NaN
1,recalibration,TF-IDF,raw,Spearman,0.674,0.604,0.733,NaN,NaN,NaN,NaN
2,recalibration,TF-IDF,raw,MAE,17.825,16.187,19.553,NaN,NaN,NaN,NaN
3,recalibration,Qwen3-8B,raw,Spearman,0.711,0.645,0.765,0.037,-0.003,0.079,0.069
4,recalibration,Qwen3-8B,raw,MAE,16.311,14.776,17.927,-1.514,-2.290,-0.743,0.000
5,recalibration,Mistral-24B,raw,Spearman,0.735,0.673,0.785,0.060,0.020,0.104,0.004
6,recalibration,Mistral-24B,raw,MAE,15.793,14.318,17.352,-2.031,-2.868,-1.159,0.000
7,recalibration,LLM-zero-shot,raw,Spearman,0.348,0.222,0.464,-0.326,-0.456,-0.195,0.000
8,recalibration,LLM-zero-shot,raw,MAE,24.377,22.531,26.221,6.552,4.959,8.129,0.000
9,recalibration,TF-IDF,recalibrated,Spearman,0.674,0.601,0.735,NaN,NaN,NaN,NaN
